# Surgery RAG Full Evaluation Pipeline

This notebook evaluates your RAG system on the ground-truth question set.

It measures retrieval, LLM answer quality, unanswerable handling, and latency.

Expected files:

```text
vector_store/surgery_logs_faiss.index
vector_store/surgery_logs_metadata.csv
vector_store/surgery_logs_embeddings.npy
evaluation/surgery_rag_ground_truth_100.csv
```

Outputs are saved in:

```text
evaluation_results/
```

## 1. Install packages

```bash
pip install pandas numpy faiss-cpu sentence-transformers rank-bm25 python-dotenv langchain-groq openpyxl tqdm
```

In [2]:
import os
import re
import json
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq

## 2. Configuration

In [ ]:
PROJECT_DIR = Path("/home/corpadm/my_project/laparoscopic-surgery-rag-assistant")

VECTOR_STORE_DIR = PROJECT_DIR / "vector_store"
FAISS_INDEX_PATH = VECTOR_STORE_DIR / "surgery_logs_faiss.index"
METADATA_PATH = VECTOR_STORE_DIR / "surgery_logs_metadata.csv"
EMBEDDINGS_PATH = VECTOR_STORE_DIR / "surgery_logs_embeddings.npy"

GROUND_TRUTH_PATH = PROJECT_DIR / "evaluation" / "surgery_rag_ground_truth_100.csv"

if not GROUND_TRUTH_PATH.exists():
    fallback = Path("/mnt/data/surgery_rag_ground_truth_eval_set/surgery_rag_ground_truth_100.csv")
    if fallback.exists():
        GROUND_TRUTH_PATH = fallback

EVAL_OUTPUT_DIR = PROJECT_DIR / "evaluation_results"
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 5
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"
GROQ_MODEL_NAME = "llama-3.1-8b-instant"
# JUDGE_MODEL_NAME = "llama-3.3-70b-versatile"

# Set True only if you want extra LLM judge calls. It costs more API calls.
RUN_LLM_JUDGE = False

print("Project:", PROJECT_DIR)
print("Ground truth:", GROUND_TRUTH_PATH)
print("Output:", EVAL_OUTPUT_DIR)

Project: /home/corpadm/my_project/laparoscopic-surgery-rag-assistant
Ground truth: /home/corpadm/my_project/laparoscopic-surgery-rag-assistant/evaluation/surgery_rag_ground_truth_100.csv
Output: /home/corpadm/my_project/laparoscopic-surgery-rag-assistant/evaluation_results


## 3. Load Groq LLM

Create `.env` in project root:

```text
GROQ_API_KEY=your_groq_api_key_here
```

In [ ]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it to .env")

llm = ChatGroq(
    model=GROQ_MODEL_NAME,
    groq_api_key=GROQ_API_KEY,
    temperature=0.1,
)

# judge_llm = ChatGroq(
#     model=JUDGE_MODEL_NAME,
#     groq_api_key=GROQ_API_KEY,
#     temperature=0.0,
# )

print("Groq model loaded:", GROQ_MODEL_NAME)

Groq model loaded: llama-3.1-8b-instant


## 4. Load vector store and ground truth

In [25]:
for path in [FAISS_INDEX_PATH, METADATA_PATH, EMBEDDINGS_PATH, GROUND_TRUTH_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

loaded_index = faiss.read_index(str(FAISS_INDEX_PATH))
loaded_metadata = pd.read_csv(METADATA_PATH)
loaded_embeddings = np.load(EMBEDDINGS_PATH)
ground_truth_df = pd.read_csv(GROUND_TRUTH_PATH)

print("FAISS vectors:", loaded_index.ntotal)
print("Metadata rows:", len(loaded_metadata))
print("Embeddings shape:", loaded_embeddings.shape)
print("Ground truth questions:", len(ground_truth_df))
display(ground_truth_df.head(10))

FAISS vectors: 509
Metadata rows: 509
Embeddings shape: (509, 768)
Ground truth questions: 100


,question_id,question,ground_truth_answer,question_type,difficulty,evaluation_focus,is_answerable,expected_case_id,expected_source_file,expected_chunk_type,expected_answer_keywords,ideal_response_behavior,notes
0,Q001,What surgery type is recorded in case Cholecys...,The surgery type for case Cholecystectomy_2025...,direct_fact,easy,metadata_retrieval,True,Cholecystectomy_2025-10-01_12-34-00,Cholecystectomy_2025-10-01_12-34-00.json,case_summary,Cholecystectomy,Answer from retrieved context only.,NaN
1,Q002,What surgery type is recorded in case Myomecto...,The surgery type for case Myomectomy_2025-10-0...,direct_fact,easy,metadata_retrieval,True,Myomectomy_2025-10-02_14-51-00,Myomectomy_2025-10-02_14-51-00.json,case_summary,Myomectomy,Answer from retrieved context only.,NaN
2,Q003,What surgery type is recorded in case Ovarian_...,The surgery type for case Ovarian_Cystectomy_2...,direct_fact,easy,metadata_retrieval,True,Ovarian_Cystectomy_2025-10-03_10-12-00,Ovarian_Cystectomy_2025-10-03_10-12-00.json,case_summary,Ovarian Cystectomy,Answer from retrieved context only.,NaN
3,Q004,What surgery type is recorded in case Partial_...,The surgery type for case Partial_or_Total_Nep...,direct_fact,easy,metadata_retrieval,True,Partial_or_Total_Nephrectomy_2025-10-04_12-48-00,Partial_or_Total_Nephrectomy_2025-10-04_12-48-...,case_summary,Partial or Total Nephrectomy,Answer from retrieved context only.,NaN
4,Q005,What surgery type is recorded in case Prostate...,The surgery type for case Prostatectomy_2025-1...,direct_fact,easy,metadata_retrieval,True,Prostatectomy_2025-10-05_11-49-00,Prostatectomy_2025-10-05_11-49-00.json,case_summary,Prostatectomy,Answer from retrieved context only.,NaN
5,Q006,Who was the surgeon for case Hernia_Repair_202...,The surgeon for case Hernia_Repair_2025-10-06_...,direct_fact,easy,metadata_retrieval,True,Hernia_Repair_2025-10-06_16-03-00,Hernia_Repair_2025-10-06_16-03-00.json,case_summary,Dr.Rao,Answer from retrieved context only.,NaN
6,Q007,Who was the surgeon for case Appendectomy_2025...,The surgeon for case Appendectomy_2025-10-07_1...,direct_fact,easy,metadata_retrieval,True,Appendectomy_2025-10-07_13-28-00,Appendectomy_2025-10-07_13-28-00.json,case_summary,Dr.Mehta,Answer from retrieved context only.,NaN
7,Q008,Who was the surgeon for case Hysterectomy_2025...,The surgeon for case Hysterectomy_2025-10-08_1...,direct_fact,easy,metadata_retrieval,True,Hysterectomy_2025-10-08_12-14-00,Hysterectomy_2025-10-08_12-14-00.json,case_summary,Dr.Meril M,Answer from retrieved context only.,NaN
8,Q009,Who was the surgeon for case Cholecystectomy_2...,The surgeon for case Cholecystectomy_2025-10-0...,direct_fact,easy,metadata_retrieval,True,Cholecystectomy_2025-10-09_11-48-00,Cholecystectomy_2025-10-09_11-48-00.json,case_summary,Dr.Mehta,Answer from retrieved context only.,NaN
9,Q010,Who was the surgeon for case Myomectomy_2025-1...,The surgeon for case Myomectomy_2025-10-10_14-...,direct_fact,easy,metadata_retrieval,True,Myomectomy_2025-10-10_14-49-00,Myomectomy_2025-10-10_14-49-00.json,case_summary,Dr.MERAI,Answer from retrieved context only.,NaN


## 5. Load embedding model

In [26]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

Embedding model loaded: all-mpnet-base-v2


## 6. Utility functions

In [27]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x)


def split_pipe_values(x):
    x = clean_text(x).strip()
    if not x:
        return []
    return [v.strip() for v in x.split("|") if v.strip()]


def normalize_for_match(x):
    return re.sub(r"\s+", " ", clean_text(x).lower().strip())


def tokenize(text):
    text = clean_text(text).lower()
    text = re.sub(r"[^a-z0-9_]+", " ", text)
    return text.split()


def normalize_scores(scores):
    scores = np.array(scores, dtype="float32")
    if len(scores) == 0:
        return scores
    mn, mx = scores.min(), scores.max()
    if mx - mn == 0:
        return np.ones_like(scores)
    return (scores - mn) / (mx - mn)


def apply_metadata_filters(df, filters=None):
    if not filters:
        return df.copy()
    filtered = df.copy()
    for col, value in filters.items():
        if value is None or col not in filtered.columns:
            continue
        if isinstance(value, str):
            filtered = filtered[filtered[col].fillna("").astype(str).str.contains(value, case=False, na=False)]
        elif isinstance(value, list):
            pattern = "|".join([str(v) for v in value])
            filtered = filtered[filtered[col].fillna("").astype(str).str.contains(pattern, case=False, na=False)]
        else:
            filtered = filtered[filtered[col] == value]
    return filtered.reset_index(drop=True)

## 7. Hybrid retrieval

This uses metadata filtering + FAISS semantic search + BM25 keyword search.

In [28]:
def hybrid_search(query, top_k=5, filters=None, semantic_weight=0.65, keyword_weight=0.35):
    start_total = time.time()
    filtered_metadata = apply_metadata_filters(loaded_metadata, filters)

    if filtered_metadata.empty:
        return pd.DataFrame(), {
            "retrieval_latency_ms": round((time.time() - start_total) * 1000, 2),
            "faiss_latency_ms": 0.0,
            "bm25_latency_ms": 0.0,
        }

    filtered_vector_ids = filtered_metadata["vector_id"].astype(int).tolist()
    filtered_embeddings = loaded_embeddings[filtered_vector_ids].astype("float32")

    temp_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
    temp_index.add(filtered_embeddings)

    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    candidate_k = min(max(top_k * 3, top_k), len(filtered_metadata))

    start_faiss = time.time()
    semantic_scores, semantic_indices = temp_index.search(query_embedding, candidate_k)
    faiss_latency_ms = round((time.time() - start_faiss) * 1000, 2)

    semantic_results = filtered_metadata.iloc[semantic_indices[0]].copy()
    semantic_results["semantic_score"] = semantic_scores[0]

    start_bm25 = time.time()
    filtered_texts = filtered_metadata["text"].fillna("").astype(str).tolist()
    filtered_tokens = [tokenize(doc) for doc in filtered_texts]
    temp_bm25 = BM25Okapi(filtered_tokens)
    keyword_scores = temp_bm25.get_scores(tokenize(query))
    bm25_latency_ms = round((time.time() - start_bm25) * 1000, 2)

    keyword_results = filtered_metadata.copy()
    keyword_results["keyword_score"] = keyword_scores
    keyword_results = keyword_results.sort_values("keyword_score", ascending=False).head(candidate_k)

    combined = pd.merge(
        semantic_results[["vector_id", "semantic_score"]],
        keyword_results[["vector_id", "keyword_score"]],
        on="vector_id",
        how="outer",
    )
    combined["semantic_score"] = combined["semantic_score"].fillna(0)
    combined["keyword_score"] = combined["keyword_score"].fillna(0)
    combined["semantic_score_norm"] = normalize_scores(combined["semantic_score"])
    combined["keyword_score_norm"] = normalize_scores(combined["keyword_score"])
    combined["hybrid_score"] = semantic_weight * combined["semantic_score_norm"] + keyword_weight * combined["keyword_score_norm"]

    final_results = pd.merge(combined, loaded_metadata, on="vector_id", how="left")
    final_results = final_results.sort_values("hybrid_score", ascending=False).head(top_k)
    final_results["rank"] = range(1, len(final_results) + 1)

    timing = {
        "retrieval_latency_ms": round((time.time() - start_total) * 1000, 2),
        "faiss_latency_ms": faiss_latency_ms,
        "bm25_latency_ms": bm25_latency_ms,
    }
    return final_results.reset_index(drop=True), timing

## 8. RAG context and prompt

In [29]:
def build_context(retrieved_df, max_chunks=5):
    if retrieved_df.empty:
        return ""
    parts = []
    for _, row in retrieved_df.head(max_chunks).iterrows():
        source = (
            f"Source: case_id={row.get('case_id')}, "
            f"chunk_id={row.get('chunk_id')}, "
            f"chunk_type={row.get('chunk_type')}, "
            f"surgery_type={row.get('surgery_type')}, "
            f"surgeon={row.get('surgeon_name')}, "
            f"source_file={row.get('source_file')}"
        )
        parts.append(f"{source}\n{row.get('text', '')}")
    return "\n\n---\n\n".join(parts)


def build_llm_prompt(question, context):
    return f"""
You are a healthcare data assistant for laparoscopic surgery event-log analysis.

Answer the user's question using only the retrieved surgery-log context.

Strict rules:
1. Do not invent information.
2. If the answer is not available in the retrieved context, say exactly:
   "Not available in the retrieved surgery logs."
3. Mention case_id and source_file when possible.
4. Keep the answer concise and useful for surgical data review.
5. Do not provide medical advice, diagnosis, or clinical decisions.
6. If multiple records are retrieved, compare only using values present in the context.

Retrieved Surgery-Log Context:
{context}

User Question:
{question}

Final Answer:
""".strip()

## 9. Run one RAG answer

In [30]:
def answer_question_with_rag(question, top_k=5, filters=None):
    total_start = time.time()

    retrieved_df, retrieval_timing = hybrid_search(question, top_k=top_k, filters=filters)
    context = build_context(retrieved_df, max_chunks=top_k)
    prompt = build_llm_prompt(question, context)

    llm_start = time.time()
    response = llm.invoke(prompt)
    llm_latency_ms = round((time.time() - llm_start) * 1000, 2)

    token_usage = {}
    try:
        token_usage = response.response_metadata.get("token_usage", {})
    except Exception:
        pass

    return {
        "answer": response.content,
        "retrieved_chunks": retrieved_df,
        "context": context,
        "prompt": prompt,
        "retrieval_latency_ms": retrieval_timing["retrieval_latency_ms"],
        "faiss_latency_ms": retrieval_timing["faiss_latency_ms"],
        "bm25_latency_ms": retrieval_timing["bm25_latency_ms"],
        "llm_latency_ms": llm_latency_ms,
        "total_latency_ms": round((time.time() - total_start) * 1000, 2),
        "token_usage": token_usage,
    }

## 10. Retrieval metrics

In [31]:
def first_hit_rank(retrieved_values, expected_values):
    if not expected_values:
        return None
    expected_norm = [normalize_for_match(x) for x in expected_values]
    for idx, value in enumerate(retrieved_values, start=1):
        if normalize_for_match(value) in expected_norm:
            return idx
    return None


def compute_retrieval_metrics(row, retrieved_df, k=5):
    expected_cases = split_pipe_values(row.get("expected_case_id", ""))
    expected_files = split_pipe_values(row.get("expected_source_file", ""))
    top = retrieved_df.head(k).copy()

    retrieved_cases = top["case_id"].fillna("").astype(str).tolist() if "case_id" in top.columns else []
    retrieved_files = top["source_file"].fillna("").astype(str).tolist() if "source_file" in top.columns else []

    case_rank = first_hit_rank(retrieved_cases, expected_cases)
    file_rank = first_hit_rank(retrieved_files, expected_files)

    has_expected = bool(expected_cases or expected_files)
    case_hit = case_rank is not None
    source_hit = file_rank is not None

    recall_at_k = 1.0 if has_expected and (case_hit or source_hit) else (0.0 if has_expected else None)

    valid_ranks = [r for r in [case_rank, file_rank] if r is not None]
    mrr = 1.0 / min(valid_ranks) if valid_ranks else (0.0 if has_expected else None)

    if has_expected:
        expected_cases_norm = set(normalize_for_match(x) for x in expected_cases)
        expected_files_norm = set(normalize_for_match(x) for x in expected_files)
        relevant_count = 0
        for _, r in top.iterrows():
            c = normalize_for_match(r.get("case_id", ""))
            f = normalize_for_match(r.get("source_file", ""))
            if c in expected_cases_norm or f in expected_files_norm:
                relevant_count += 1
        precision_at_k = relevant_count / max(len(top), 1)
    else:
        precision_at_k = None

    return {
        "case_hit_at_k": case_hit,
        "source_hit_at_k": source_hit,
        "recall_at_k": recall_at_k,
        "precision_at_k": precision_at_k,
        "mrr": mrr,
        "case_rank": case_rank,
        "source_rank": file_rank,
        "retrieved_count": len(top),
    }

## 11. LLM answer metrics

In [32]:
def keyword_coverage(answer, expected_keywords):
    keywords = split_pipe_values(expected_keywords)
    if not keywords:
        return None
    ans = normalize_for_match(answer)
    matched, missed = [], []
    for kw in keywords:
        if normalize_for_match(kw) in ans:
            matched.append(kw)
        else:
            missed.append(kw)
    return {
        "keyword_coverage": len(matched) / len(keywords),
        "matched_keywords": matched,
        "missed_keywords": missed,
    }


def contains_not_available(answer):
    ans = normalize_for_match(answer)
    patterns = [
        "not available in the retrieved surgery logs",
        "not available",
        "not found",
        "not present",
        "cannot determine",
        "no matching surgery logs",
    ]
    return any(p in ans for p in patterns)


def simple_answer_metrics(row, answer):
    is_answerable = bool(row.get("is_answerable", True))
    kw = keyword_coverage(answer, row.get("expected_answer_keywords", ""))
    kw_score = None if kw is None else kw["keyword_coverage"]
    matched = [] if kw is None else kw["matched_keywords"]
    missed = [] if kw is None else kw["missed_keywords"]
    not_available = contains_not_available(answer)

    return {
        "keyword_coverage": kw_score,
        "matched_keywords": "|".join(matched),
        "missed_keywords": "|".join(missed),
        "contains_not_available": not_available,
        "answerable_behavior_correct": (not not_available) if is_answerable else None,
        "unanswerable_handling_correct": not_available if not is_answerable else None,
    }

## 12. Optional LLM-as-judge metrics

Set `RUN_LLM_JUDGE = True` in configuration to enable this.

It adds extra API calls.

In [33]:
def safe_json_parse(text):
    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except Exception:
                return {}
    return {}


def llm_judge_evaluate(question, ground_truth, generated_answer, context):
    judge_prompt = f"""
You are an evaluator for a Retrieval-Augmented Generation system.

Return only valid JSON:
{{
  "faithfulness": number from 0 to 1,
  "answer_correctness": number from 0 to 1,
  "answer_relevance": number from 0 to 1,
  "hallucination_risk": number from 0 to 1,
  "reason": "short explanation"
}}

Question:
{question}

Ground Truth:
{ground_truth}

Retrieved Context:
{context}

Generated Answer:
{generated_answer}
"""
    response = judge_llm.invoke(judge_prompt)
    parsed = safe_json_parse(response.content)
    return {
        "judge_faithfulness": parsed.get("faithfulness"),
        "judge_answer_correctness": parsed.get("answer_correctness"),
        "judge_answer_relevance": parsed.get("answer_relevance"),
        "judge_hallucination_risk": parsed.get("hallucination_risk"),
        "judge_reason": parsed.get("reason", ""),
    }

## 13. Run evaluation on all questions

For quick test:

```python
MAX_QUESTIONS = 5
```

For full evaluation:

```python
MAX_QUESTIONS = None
```

In [36]:
MAX_QUESTIONS = None

eval_df = ground_truth_df.copy()
if MAX_QUESTIONS is not None:
    eval_df = eval_df.head(MAX_QUESTIONS)

results = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    question = row["question"]
    ground_truth = row["ground_truth_answer"]

    try:
        rag_output = answer_question_with_rag(question, top_k=TOP_K, filters=None)
        answer = rag_output["answer"]
        retrieved_df = rag_output["retrieved_chunks"]

        retrieval_metrics = compute_retrieval_metrics(row, retrieved_df, k=TOP_K)
        answer_metrics = simple_answer_metrics(row, answer)

        retrieved_case_ids = retrieved_df["case_id"].fillna("").astype(str).tolist() if "case_id" in retrieved_df.columns else []
        retrieved_source_files = retrieved_df["source_file"].fillna("").astype(str).tolist() if "source_file" in retrieved_df.columns else []
        retrieved_chunk_ids = retrieved_df["chunk_id"].fillna("").astype(str).tolist() if "chunk_id" in retrieved_df.columns else []
        retrieved_chunk_types = retrieved_df["chunk_type"].fillna("").astype(str).tolist() if "chunk_type" in retrieved_df.columns else []

        judge_metrics = {}
        if RUN_LLM_JUDGE:
            judge_metrics = llm_judge_evaluate(question, ground_truth, answer, rag_output["context"])

        token_usage = rag_output.get("token_usage", {}) or {}

        result = {
            "question_id": row.get("question_id", ""),
            "question": question,
            "ground_truth_answer": ground_truth,
            "generated_answer": answer,
            "question_type": row.get("question_type", ""),
            "difficulty": row.get("difficulty", ""),
            "evaluation_focus": row.get("evaluation_focus", ""),
            "is_answerable": row.get("is_answerable", True),
            "expected_case_id": row.get("expected_case_id", ""),
            "expected_source_file": row.get("expected_source_file", ""),
            "expected_chunk_type": row.get("expected_chunk_type", ""),
            "expected_answer_keywords": row.get("expected_answer_keywords", ""),
            "retrieved_case_ids": "|".join(retrieved_case_ids),
            "retrieved_source_files": "|".join(retrieved_source_files),
            "retrieved_chunk_ids": "|".join(retrieved_chunk_ids),
            "retrieved_chunk_types": "|".join(retrieved_chunk_types),
            "retrieval_latency_ms": rag_output["retrieval_latency_ms"],
            "faiss_latency_ms": rag_output["faiss_latency_ms"],
            "bm25_latency_ms": rag_output["bm25_latency_ms"],
            "llm_latency_ms": rag_output["llm_latency_ms"],
            "total_latency_ms": rag_output["total_latency_ms"],
            "prompt_tokens": token_usage.get("prompt_tokens"),
            "completion_tokens": token_usage.get("completion_tokens"),
            "total_tokens": token_usage.get("total_tokens"),
            "error": "",
        }
        result.update(retrieval_metrics)
        result.update(answer_metrics)
        result.update(judge_metrics)
        results.append(result)

    except Exception as e:
        results.append({
            "question_id": row.get("question_id", ""),
            "question": question,
            "ground_truth_answer": ground_truth,
            "generated_answer": "",
            "question_type": row.get("question_type", ""),
            "difficulty": row.get("difficulty", ""),
            "evaluation_focus": row.get("evaluation_focus", ""),
            "is_answerable": row.get("is_answerable", True),
            "error": str(e),
        })

results_df = pd.DataFrame(results)
display(results_df.head())
print("Completed:", len(results_df))
print("Errors:", (results_df["error"].fillna("") != "").sum())

100%|██████████| 100/100 [15:38<00:00,  9.39s/it]


,question_id,question,ground_truth_answer,generated_answer,question_type,difficulty,evaluation_focus,is_answerable,expected_case_id,expected_source_file,...,mrr,case_rank,source_rank,retrieved_count,keyword_coverage,matched_keywords,missed_keywords,contains_not_available,answerable_behavior_correct,unanswerable_handling_correct
0,Q001,What surgery type is recorded in case Cholecys...,The surgery type for case Cholecystectomy_2025...,Cholecystectomy. (Source: case_id=Cholecystect...,direct_fact,easy,metadata_retrieval,True,Cholecystectomy_2025-10-01_12-34-00,Cholecystectomy_2025-10-01_12-34-00.json,...,1.0,1.0,1.0,5.0,1.0,Cholecystectomy,,False,True,NaN
1,Q002,What surgery type is recorded in case Myomecto...,The surgery type for case Myomectomy_2025-10-0...,Myomectomy.,direct_fact,easy,metadata_retrieval,True,Myomectomy_2025-10-02_14-51-00,Myomectomy_2025-10-02_14-51-00.json,...,1.0,1.0,1.0,5.0,1.0,Myomectomy,,False,True,NaN
2,Q003,What surgery type is recorded in case Ovarian_...,The surgery type for case Ovarian_Cystectomy_2...,The surgery type recorded in case Ovarian_Cyst...,direct_fact,easy,metadata_retrieval,True,Ovarian_Cystectomy_2025-10-03_10-12-00,Ovarian_Cystectomy_2025-10-03_10-12-00.json,...,1.0,1.0,1.0,5.0,1.0,Ovarian Cystectomy,,False,True,NaN
3,Q004,What surgery type is recorded in case Partial_...,The surgery type for case Partial_or_Total_Nep...,The surgery type recorded in case Partial_or_T...,direct_fact,easy,metadata_retrieval,True,Partial_or_Total_Nephrectomy_2025-10-04_12-48-00,Partial_or_Total_Nephrectomy_2025-10-04_12-48-...,...,1.0,1.0,1.0,5.0,1.0,Partial or Total Nephrectomy,,False,True,NaN
4,Q005,What surgery type is recorded in case Prostate...,The surgery type for case Prostatectomy_2025-1...,The surgery type recorded in case Prostatectom...,direct_fact,easy,metadata_retrieval,True,Prostatectomy_2025-10-05_11-49-00,Prostatectomy_2025-10-05_11-49-00.json,...,1.0,1.0,1.0,5.0,1.0,Prostatectomy,,False,True,NaN


Completed: 100
Errors: 22


## 14. Summary metrics

In [37]:
def safe_mean(series):
    s = pd.to_numeric(series, errors="coerce")
    return None if s.dropna().empty else float(s.mean())


def safe_percent(series):
    s = series.dropna()
    return None if len(s) == 0 else float(s.astype(float).mean() * 100)

answerable_df = results_df[results_df["is_answerable"] == True].copy()
unanswerable_df = results_df[results_df["is_answerable"] == False].copy()

summary = {
    "evaluation_timestamp": datetime.now().isoformat(),
    "top_k": TOP_K,
    "total_questions": int(len(results_df)),
    "answerable_questions": int(len(answerable_df)),
    "unanswerable_questions": int(len(unanswerable_df)),
    "errors": int((results_df["error"].fillna("") != "").sum()),
    "retrieval_recall_at_k_percent": safe_percent(results_df["recall_at_k"]),
    "retrieval_precision_at_k_percent": safe_percent(results_df["precision_at_k"]),
    "retrieval_mrr": safe_mean(results_df["mrr"]),
    "case_hit_at_k_percent": safe_percent(results_df["case_hit_at_k"]),
    "source_hit_at_k_percent": safe_percent(results_df["source_hit_at_k"]),
    "avg_keyword_coverage_percent": safe_percent(results_df["keyword_coverage"]),
    "answerable_behavior_accuracy_percent": safe_percent(answerable_df["answerable_behavior_correct"]),
    "unanswerable_handling_accuracy_percent": safe_percent(unanswerable_df["unanswerable_handling_correct"]),
    "avg_retrieval_latency_ms": safe_mean(results_df["retrieval_latency_ms"]),
    "p95_retrieval_latency_ms": float(pd.to_numeric(results_df["retrieval_latency_ms"], errors="coerce").quantile(0.95)),
    "avg_faiss_latency_ms": safe_mean(results_df["faiss_latency_ms"]),
    "avg_bm25_latency_ms": safe_mean(results_df["bm25_latency_ms"]),
    "avg_llm_latency_ms": safe_mean(results_df["llm_latency_ms"]),
    "p95_llm_latency_ms": float(pd.to_numeric(results_df["llm_latency_ms"], errors="coerce").quantile(0.95)),
    "avg_total_latency_ms": safe_mean(results_df["total_latency_ms"]),
    "p95_total_latency_ms": float(pd.to_numeric(results_df["total_latency_ms"], errors="coerce").quantile(0.95)),
}

if RUN_LLM_JUDGE and "judge_faithfulness" in results_df.columns:
    summary.update({
        "judge_avg_faithfulness": safe_mean(results_df["judge_faithfulness"]),
        "judge_avg_answer_correctness": safe_mean(results_df["judge_answer_correctness"]),
        "judge_avg_answer_relevance": safe_mean(results_df["judge_answer_relevance"]),
        "judge_avg_hallucination_risk": safe_mean(results_df["judge_hallucination_risk"]),
    })

summary_df = pd.DataFrame([summary]).T.reset_index()
summary_df.columns = ["metric", "value"]
display(summary_df)

,metric,value
0,evaluation_timestamp,2026-08-14T19:39:25.545474
1,top_k,5
2,total_questions,100
3,answerable_questions,80
4,unanswerable_questions,20
5,errors,22
6,retrieval_recall_at_k_percent,84.0
7,retrieval_precision_at_k_percent,37.333333
8,retrieval_mrr,0.831111
9,case_hit_at_k_percent,80.769231


## 15. Breakdown by question type

In [38]:
breakdown_rows = []
for qtype, group in results_df.groupby("question_type"):
    breakdown_rows.append({
        "question_type": qtype,
        "count": len(group),
        "recall_at_k_percent": safe_percent(group["recall_at_k"]),
        "precision_at_k_percent": safe_percent(group["precision_at_k"]),
        "mrr": safe_mean(group["mrr"]),
        "keyword_coverage_percent": safe_percent(group["keyword_coverage"]),
        "unanswerable_accuracy_percent": safe_percent(group["unanswerable_handling_correct"]) if "unanswerable_handling_correct" in group else None,
        "avg_retrieval_latency_ms": safe_mean(group["retrieval_latency_ms"]),
        "avg_llm_latency_ms": safe_mean(group["llm_latency_ms"]),
        "avg_total_latency_ms": safe_mean(group["total_latency_ms"]),
    })

breakdown_df = pd.DataFrame(breakdown_rows).sort_values("question_type")
display(breakdown_df)

,question_type,count,recall_at_k_percent,precision_at_k_percent,mrr,keyword_coverage_percent,unanswerable_accuracy_percent,avg_retrieval_latency_ms,avg_llm_latency_ms,avg_total_latency_ms
0,comparison,10,100.000000,44.000000,1.000000,10.000000,None,59.821000,12258.568000,12319.728000
1,corpus_aggregation,10,0.000000,0.000000,0.000000,11.111111,None,40.382222,13465.900000,13507.275556
2,direct_fact,25,96.000000,40.000000,0.960000,96.000000,None,37.886800,9412.936000,9451.803600
3,filtered_case_lookup,6,16.666667,3.333333,0.055556,0.000000,None,39.816667,14891.658333,14932.520000
4,filtered_instrument,8,100.000000,47.500000,1.000000,43.750000,None,53.346250,10993.538750,11048.263750
5,instrument_list,10,100.000000,42.000000,1.000000,32.523810,None,37.117000,11552.807000,11591.047000
6,instrument_max,5,100.000000,52.000000,1.000000,20.000000,None,42.964000,12973.440000,13017.518000
7,instrument_side_grouping,5,100.000000,56.000000,1.000000,0.000000,None,38.838000,10755.286000,10795.168000
8,paraphrased_instrument_max,1,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
9,unanswerable_missing_field,10,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN


## 16. Failure analysis

In [40]:
failure_mask = pd.Series(False, index=results_df.index)

if "recall_at_k" in results_df:
    failure_mask |= results_df["recall_at_k"].fillna(1) == 0
if "keyword_coverage" in results_df:
    failure_mask |= results_df["keyword_coverage"].fillna(1) < 0.5
if "unanswerable_handling_correct" in results_df:
    failure_mask |= results_df["unanswerable_handling_correct"].fillna(True) == False
if "answerable_behavior_correct" in results_df:
    failure_mask |= results_df["answerable_behavior_correct"].fillna(True) == False
if "error" in results_df:
    failure_mask |= results_df["error"].fillna("") != ""

failures_df = results_df[failure_mask].copy()
print("Potential failures:", len(failures_df))

cols = [
    "question_id", "question", "question_type", "is_answerable",
    "ground_truth_answer", "generated_answer",
    "recall_at_k", "precision_at_k", "mrr", "keyword_coverage",
    "unanswerable_handling_correct", "answerable_behavior_correct",
    "expected_case_id", "retrieved_case_ids",
    "expected_source_file", "retrieved_source_files",
    "missed_keywords", "error",
]
cols = [c for c in cols if c in failures_df.columns]
display(failures_df[cols].head(10))

Potential failures: 64


/tmp/ipykernel_16533/244822852.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  failure_mask |= results_df["unanswerable_handling_correct"].fillna(True) == False
/tmp/ipykernel_16533/244822852.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  failure_mask |= results_df["answerable_behavior_correct"].fillna(True) == False


,question_id,question,question_type,is_answerable,ground_truth_answer,generated_answer,recall_at_k,precision_at_k,mrr,keyword_coverage,unanswerable_handling_correct,answerable_behavior_correct,expected_case_id,retrieved_case_ids,expected_source_file,retrieved_source_files,missed_keywords,error
9,Q010,Who was the surgeon for case Myomectomy_2025-1...,direct_fact,True,The surgeon for case Myomectomy_2025-10-10_14-...,Not available in the retrieved surgery logs.,0.0,0.0,0.0,0.000000,NaN,False,Myomectomy_2025-10-10_14-49-00,Myomectomy_2025-11-14_08-27-00|Myomectomy_2025...,Myomectomy_2025-10-10_14-49-00.json,Myomectomy_2025-11-14_08-27-00.json|Myomectomy...,Dr.MERAI,
25,Q026,Which instruments were used in case Myomectomy...,instrument_list,True,The instruments used in case Myomectomy_2025-1...,Not available in the retrieved surgery logs.,1.0,0.2,1.0,0.000000,NaN,False,Myomectomy_2025-10-26_17-36-00,Myomectomy_2025-10-26_17-36-00|Myomectomy_2025...,Myomectomy_2025-10-26_17-36-00.json,Myomectomy_2025-10-26_17-36-00.json|Myomectomy...,Endowrist_Stapler_30_Instrument|Long_Bipolar_G...,
26,Q027,Which instruments were used in case Ovarian_Cy...,instrument_list,True,The instruments used in case Ovarian_Cystectom...,The instruments used in case Ovarian_Cystectom...,1.0,0.6,1.0,0.375000,NaN,True,Ovarian_Cystectomy_2025-10-27_11-04-00,Ovarian_Cystectomy_2025-10-27_11-04-00|Ovarian...,Ovarian_Cystectomy_2025-10-27_11-04-00.json,Ovarian_Cystectomy_2025-10-27_11-04-00.json|Ov...,Atrial_Retractor_Short_Right|Dual_Blade_Retrac...,
27,Q028,Which instruments were used in case Partial_or...,instrument_list,True,The instruments used in case Partial_or_Total_...,"Based on the retrieved surgery logs, the instr...",1.0,0.4,1.0,0.125000,NaN,True,Partial_or_Total_Nephrectomy_2025-10-28_09-49-00,Partial_or_Total_Nephrectomy_2025-10-28_09-49-...,Partial_or_Total_Nephrectomy_2025-10-28_09-49-...,Partial_or_Total_Nephrectomy_2025-10-28_09-49-...,Cadiere_Forceps|Long_Bipolar_Grasper|Maryland_...,
28,Q029,Which instruments were used in case Prostatect...,instrument_list,True,The instruments used in case Prostatectomy_202...,Not available in the retrieved surgery logs.,1.0,0.2,1.0,0.000000,NaN,False,Prostatectomy_2025-11-01_14-48-00,Prostatectomy_2025-11-01_14-48-00|Prostatectom...,Prostatectomy_2025-11-01_14-48-00.json,Prostatectomy_2025-11-01_14-48-00.json|Prostat...,Cadiere_Forceps|Endowrist_Stapler_30_Instrumen...,
29,Q030,Which instruments were used in case Hernia_Rep...,instrument_list,True,The instruments used in case Hernia_Repair_202...,Two instruments were used in case Hernia_Repai...,1.0,0.4,1.0,0.285714,NaN,True,Hernia_Repair_2025-11-02_11-03-00,Hernia_Repair_2025-11-02_11-03-00|Hernia_Repai...,Hernia_Repair_2025-11-02_11-03-00.json,Hernia_Repair_2025-11-02_11-03-00.json|Hernia_...,Black_Diamond_Micro_Forceps|Maryland_Bipolar_F...,
30,Q031,Which instruments were used in case Appendecto...,instrument_list,True,The instruments used in case Appendectomy_2025...,"Based on the retrieved surgery logs, the instr...",1.0,0.6,1.0,0.333333,NaN,True,Appendectomy_2025-11-03_09-34-00,Appendectomy_2025-11-03_09-34-00|Appendectomy_...,Appendectomy_2025-11-03_09-34-00.json,Appendectomy_2025-11-03_09-34-00.json|Appendec...,Black_Diamond_Micro_Forceps|Maryland_Bipolar_F...,
33,Q034,Which instruments were used in case Myomectomy...,instrument_list,True,The instruments used in case Myomectomy_2025-1...,The instruments used in case Myomectomy_2025-1...,1.0,0.6,1.0,0.333333,NaN,True,Myomectomy_2025-11-06_14-15-00,Myomectomy_2025-11-06_14-15-00|Myomectomy_2025...,Myomectomy_2025-11-06_14-15-00.json,Myomectomy_2025-11-06_14-15-00.json|Myomectomy...,Black_Diamond_Micro_Forceps|Force_Bipolar|Mono...,
34,Q035,Which instruments were used in case Ovarian_Cy...,instrument_list,True,The instruments used in case Ovarian_Cystectom...,"Based on the retrieved surgery-log context, th...",1.0,0.2,1.0,0.200000,NaN,True,Ovarian_Cystectomy_2025-11-07_17-50-00,Ovarian_Cystectomy_2025-11-07

In [21]:
expected = "Cholecystectomy_2025-10-01_12-34-00"

loaded_metadata[
    loaded_metadata["case_id"].astype(str).str.contains(expected, case=False, na=False)
][["case_id", "source_file", "chunk_type", "text"]].head()

,case_id,source_file,chunk_type,text


## 17. Save reports

In [19]:
detailed_path = EVAL_OUTPUT_DIR / "rag_eval_detailed_results.csv"
summary_path = EVAL_OUTPUT_DIR / "rag_eval_summary.csv"
breakdown_path = EVAL_OUTPUT_DIR / "rag_eval_breakdown_by_type.csv"
failures_path = EVAL_OUTPUT_DIR / "rag_eval_failures.csv"
summary_json_path = EVAL_OUTPUT_DIR / "rag_eval_summary.json"
excel_path = EVAL_OUTPUT_DIR / "rag_eval_report.xlsx"

results_df.to_csv(detailed_path, index=False)
summary_df.to_csv(summary_path, index=False)
breakdown_df.to_csv(breakdown_path, index=False)
failures_df.to_csv(failures_path, index=False)

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="detailed_results", index=False)
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    breakdown_df.to_excel(writer, sheet_name="breakdown_by_type", index=False)
    failures_df.to_excel(writer, sheet_name="failures", index=False)

print("Saved:")
print(detailed_path)
print(summary_path)
print(breakdown_path)
print(failures_path)
print(summary_json_path)
print(excel_path)

ModuleNotFoundError: No module named 'openpyxl'

## 18. Resume / GitHub claim template

Use this only after running full evaluation:

```text
Evaluated the surgery-log RAG system on 100 ground-truth queries covering direct fact, metadata-filtered, instrument, comparison, aggregation, and unanswerable edge-case questions. Measured Recall@5, Precision@5, MRR, source-hit accuracy, keyword coverage, unanswerable-handling accuracy, retrieval latency, LLM latency, and end-to-end latency.
```

Use actual metric values from `evaluation_results/rag_eval_summary.csv`.